# March Machine Learning Mania 2026 — Bracket Explorer
Win probabilities generated by XGBoost models trained on per-team efficiency, Elo, momentum, Q1 record, and variance features.

**Monte Carlo simulation**: the tournament is simulated N times using model win probabilities at every game. Results show each team's probability of reaching each round.

In [55]:
import pandas as pd
import numpy as np

RAW        = r"C:\Personal Projects\MMLM-2026\data\raw"
PROCESSED  = r"C:\Personal Projects\MMLM-2026\data\processed"
SUBMISSION = r"C:\Personal Projects\MMLM-2026\submissions\submission.csv"
SEASON     = 2026
N_SIMS     = 10000  # increase for smoother probabilities, decrease to run faster

submission = pd.read_csv(SUBMISSION)
m_features = pd.read_csv(f"{PROCESSED}/M_team_features.csv")
w_features = pd.read_csv(f"{PROCESSED}/W_team_features.csv")
teams_m    = pd.read_csv(f"{RAW}/MTeams.csv")
teams_w    = pd.read_csv(f"{RAW}/WTeams.csv")

print(f"Submission rows: {len(submission):,}")
print(f"Pred range: [{submission['Pred'].min():.4f}, {submission['Pred'].max():.4f}]")
print(f"Pred mean:  {submission['Pred'].mean():.4f}")

Submission rows: 132,133
Pred range: [0.0250, 0.9750]
Pred mean:  0.5010


In [56]:
# -----------------------------------------------------------------------
# Core helpers
# -----------------------------------------------------------------------

# Pre-build lookup dict — makes simulation ~100x faster
prob_lookup = {row['ID']: row['Pred'] for _, row in submission.iterrows()}

def get_prob(team_a, team_b, season=SEASON):
    """Returns P(team_a beats team_b). O(1) dict lookup."""
    if team_a < team_b:
        return prob_lookup.get(f"{season}_{team_a}_{team_b}", 0.5)
    else:
        return 1 - prob_lookup.get(f"{season}_{team_b}_{team_a}", 0.5)


def get_seeds(features, teams, season=SEASON):
    """Build seedings table. Play-in teams share the same SeedNum (e.g. two teams at SeedNum=11)."""
    return (features[features['Season'] == season][['TeamID', 'SeedNum', 'Region']]
            .merge(teams[['TeamID', 'TeamName']], on='TeamID')
            .sort_values(['Region', 'SeedNum']))


def resolve_playin(seeds, region, seed_num, rng):
    """
    For a given seed slot, check if two teams are competing for it (play-in).
    If so, simulate the play-in game and return the winner's TeamID.
    If only one team holds the slot, return it directly.
    """
    candidates = seeds[(seeds['Region'] == region) & (seeds['SeedNum'].astype(int) == seed_num)]
    if len(candidates) == 1:
        return candidates['TeamID'].values[0]
    elif len(candidates) == 2:
        id_a = candidates['TeamID'].values[0]
        id_b = candidates['TeamID'].values[1]
        return id_a if rng.random() < get_prob(id_a, id_b) else id_b
    return None


# Standard bracket order within each region
FIRST_ROUND_PAIRS = [(1,16),(8,9),(5,12),(4,13),(6,11),(3,14),(7,10),(2,15)]

print("Core helpers ready.")

Core helpers ready.


In [57]:
# -----------------------------------------------------------------------
# Monte Carlo simulation engine
# -----------------------------------------------------------------------

def simulate_tournament(seeds, n_sims=N_SIMS, rand_seed=42):
    """
    Simulate the full tournament n_sims times using model win probabilities.
    
    Returns a DataFrame with each team's probability of reaching each round:
        P_R1    — First Round (i.e. made it to the round of 64)
        P_R2    — Second Round (round of 32)
        P_S16   — Sweet 16
        P_E8    — Elite Eight
        P_FF    — Final Four
        P_Final — Championship game
        P_Champ — Champion
    """
    rng = np.random.default_rng(rand_seed)
    regions = sorted(seeds['Region'].unique())
    all_teams = seeds['TeamID'].unique()
    round_counts = {tid: np.zeros(7) for tid in all_teams}

    for _ in range(n_sims):
        region_ff_teams = []

        for region in regions:
            # Resolve play-ins and build first round matchups
            bracket = []
            for high_seed, low_seed in FIRST_ROUND_PAIRS:
                team_a = resolve_playin(seeds, region, high_seed, rng)
                team_b = resolve_playin(seeds, region, low_seed, rng)
                if team_a is not None and team_b is not None:
                    bracket.append((team_a, team_b))

            # Simulate rounds within region until one team remains
            current = bracket
            round_idx = 0
            while len(current) > 1:
                winners = []
                for team_a, team_b in current:
                    round_counts[team_a][round_idx] += 1
                    round_counts[team_b][round_idx] += 1
                    winner = team_a if rng.random() < get_prob(team_a, team_b) else team_b
                    winners.append(winner)
                current = [(winners[i], winners[i+1]) for i in range(0, len(winners), 2)]
                round_idx += 1

            # Last matchup in region = Elite Eight
            team_a, team_b = current[0]
            round_counts[team_a][round_idx] += 1
            round_counts[team_b][round_idx] += 1
            ff_team = team_a if rng.random() < get_prob(team_a, team_b) else team_b
            round_counts[ff_team][4] += 1
            region_ff_teams.append(ff_team)

        # Final Four — pair regions and simulate semis
        ff_pairs = [(region_ff_teams[0], region_ff_teams[1]),
                    (region_ff_teams[2], region_ff_teams[3])]
        finalists = []
        for team_a, team_b in ff_pairs:
            round_counts[team_a][4] += 1
            round_counts[team_b][4] += 1
            winner = team_a if rng.random() < get_prob(team_a, team_b) else team_b
            round_counts[winner][5] += 1
            finalists.append(winner)

        # Championship
        round_counts[finalists[0]][5] += 1
        round_counts[finalists[1]][5] += 1
        champion = finalists[0] if rng.random() < get_prob(finalists[0], finalists[1]) else finalists[1]
        round_counts[champion][6] += 1

    # Build results DataFrame
    results = []
    for tid in all_teams:
        p = round_counts[tid] / n_sims
        info = seeds[seeds['TeamID'] == tid].iloc[0]
        results.append({
            'TeamID':   tid,
            'TeamName': info['TeamName'],
            'Region':   info['Region'],
            'SeedNum':  info['SeedNum'],
            'P_R1':     p[0], 'P_R2':    p[1], 'P_S16':  p[2],
            'P_E8':     p[3], 'P_FF':    p[4], 'P_Final':p[5], 'P_Champ':p[6],
        })

    return pd.DataFrame(results).sort_values(['Region', 'SeedNum']).reset_index(drop=True)

print("Simulation engine ready.")

Simulation engine ready.


In [58]:
# -----------------------------------------------------------------------
# Display helpers
# -----------------------------------------------------------------------

def display_region(sim_results, region):
    r = sim_results[sim_results['Region'] == region].sort_values('SeedNum')
    print(f"\n{'─'*75}")
    print(f"  {region} Region")
    print(f"{'─'*75}")
    print(f"  {'Seed':<6} {'Team':<22} {'R1':>6} {'R2':>6} {'S16':>6} {'E8':>6} {'FF':>6} {'Final':>6} {'Champ':>6}")
    print(f"  {'─'*72}")
    prev_pair = None
    pair_map = {1:16,16:1,8:9,9:8,5:12,12:5,4:13,13:4,6:11,11:6,3:14,14:3,7:10,10:7,2:15,15:2}
    for _, row in r.iterrows():
        s = int(row['SeedNum'])
        pair = tuple(sorted([s, pair_map.get(s, s)]))
        if prev_pair is not None and pair != prev_pair:
            print()
        seed_str = f"({s})"
        print(f"  {seed_str:<6} {row['TeamName']:<22} "
              f"{row['P_R1']:>5.1%} {row['P_R2']:>6.1%} {row['P_S16']:>6.1%} "
              f"{row['P_E8']:>6.1%} {row['P_FF']:>6.1%} {row['P_Final']:>6.1%} {row['P_Champ']:>6.1%}")
        prev_pair = pair


def display_championship_odds(sim_results, label, top_n=20):
    top = sim_results.sort_values('P_Champ', ascending=False).head(top_n)
    print(f"\n{'='*65}")
    print(f"  {label} — Championship Contenders (top {top_n})")
    print(f"{'='*65}")
    print(f"  {'Seed':<6} {'Team':<22} {'Region':<8} {'FF':>6} {'Final':>6} {'Champ':>6}")
    print(f"  {'─'*60}")
    for _, row in top.iterrows():
        print(f"  ({int(row['SeedNum'])}){'':<3} {row['TeamName']:<22} {row['Region']:<8} "
              f"{row['P_FF']:>6.1%} {row['P_Final']:>6.1%} {row['P_Champ']:>6.1%}")


def display_cinderellas(sim_results, label, min_seed=10, threshold=0.20):
    """Teams seeded >= min_seed with a meaningful Sweet 16 probability."""
    cin = (sim_results[sim_results['SeedNum'] >= min_seed]
           .query('P_S16 >= @threshold')
           .sort_values('P_S16', ascending=False))
    print(f"\n{'='*65}")
    print(f"  {label} — Cinderella Watch (seed >= {min_seed}, P(Sweet 16) >= {threshold:.0%})")
    print(f"{'='*65}")
    if len(cin) == 0:
        print("  None found at this threshold.")
        return
    print(f"  {'Seed':<6} {'Team':<22} {'Region':<8} {'R1':>6} {'S16':>6} {'E8':>6}")
    print(f"  {'─'*58}")
    for _, row in cin.iterrows():
        print(f"  ({int(row['SeedNum'])}){'':<3} {row['TeamName']:<22} {row['Region']:<8} "
              f"{row['P_R1']:>6.1%} {row['P_S16']:>6.1%} {row['P_E8']:>6.1%}")

print("Display helpers ready.")

Display helpers ready.


---
## Men's Bracket

In [59]:
print(f"Running {N_SIMS:,} simulations for Men's bracket...")
sim_m = simulate_tournament(seeds_m, n_sims=N_SIMS)
print("Done.")

Running 10,000 simulations for Men's bracket...
Done.


In [50]:
for region in sorted(seeds_m['Region'].unique()):
    display_region(sim_m, region)


───────────────────────────────────────────────────────────────────────────
  W Region
───────────────────────────────────────────────────────────────────────────
  Seed   Team                       R1     R2    S16     E8     FF  Final  Champ
  ────────────────────────────────────────────────────────────────────────
  (1)    Duke                   100.0%  95.8%  92.1%  75.6% 121.4%  93.6%  25.4%

  (2)    Connecticut            100.0%  96.4%  73.1%  38.7%  20.4%  10.2%   1.9%

  (3)    Michigan St            100.0%  91.8%  53.7%  27.7%  13.4%   5.4%   0.6%

  (4)    Kansas                 100.0%  94.9%  24.7%   3.0%   2.8%   1.0%   0.1%

  (5)    St John's              100.0%  86.5%  71.8%  19.5%  18.8%   7.8%   0.9%

  (6)    Louisville             100.0%  35.6%  19.0%  11.6%  12.6%   3.6%   0.5%

  (7)    UCLA                   100.0%  74.8%  23.5%  11.1%   3.6%   0.6%   0.0%

  (8)    Ohio St                100.0%  59.9%   5.4%   1.0%   0.6%   0.0%   0.0%
  (9)    TCU             

In [51]:
display_championship_odds(sim_m, "Men's Bracket")


  Men's Bracket — Championship Contenders (top 20)
  Seed   Team                   Region       FF  Final  Champ
  ────────────────────────────────────────────────────────────
  (1)    Duke                   W        121.4%  93.6%  25.4%
  (1)    Michigan               Y         96.4%  66.6%  21.1%
  (1)    Arizona                Z         80.4%  41.2%  10.8%
  (1)    Florida                X         87.6%  31.4%   8.8%
  (2)    Houston                X         53.0%  21.0%   6.2%
  (3)    Gonzaga                Z         34.8%  17.8%   4.8%
  (2)    Iowa St                Y         34.4%  16.4%   4.3%
  (2)    Purdue                 Z         45.0%  21.0%   2.9%
  (3)    Virginia               Y         31.6%  11.2%   2.3%
  (3)    Illinois               X         18.2%   8.8%   2.1%
  (2)    Connecticut            W         20.4%  10.2%   1.9%
  (4)    Nebraska               X         13.4%   7.0%   1.1%
  (5)    Texas Tech             Y         14.0%   5.8%   1.1%
  (5)    St John'

In [52]:
display_cinderellas(sim_m, "Men's Bracket")


  Men's Bracket — Cinderella Watch (seed >= 10, P(Sweet 16) >= 20%)
  Seed   Team                   Region       R1    S16     E8
  ──────────────────────────────────────────────────────────
  (11)    South Florida          W        100.0%  26.7%   9.3%
  (11)    VCU                    X        100.0%  24.4%   3.2%


---
## Women's Bracket

In [53]:
seeds_w = get_seeds(w_features, teams_w)
print(f"Women's 2026 tournament teams: {len(seeds_w)}")
print(f"Regions: {sorted(seeds_w['Region'].unique())}")

playin_w = seeds_w[seeds_w.duplicated(subset=['Region','SeedNum'], keep=False)]
if len(playin_w) > 0:
    print(f"\nPlay-in teams:")
    print(playin_w[['Region','SeedNum','TeamName']].to_string(index=False))
else:
    print("No play-in teams detected.")

Women's 2026 tournament teams: 68
Regions: ['W', 'X', 'Y', 'Z']

Play-in teams:
Region  SeedNum      TeamName
     X     10.5    Arizona St
     X     10.5      Virginia
     X     16.5       Samford
     X     16.5 Southern Univ
     Y     16.5   Missouri St
     Y     16.5     SF Austin
     Z     11.5      Nebraska
     Z     11.5      Richmond


In [54]:
print(f"Running {N_SIMS:,} simulations for Women's bracket...")
sim_w = simulate_tournament(seeds_w, n_sims=N_SIMS)
print("Done.")

Running 1,000 simulations for Women's bracket...
Done.


In [42]:
for region in sorted(seeds_w['Region'].unique()):
    display_region(sim_w, region)


───────────────────────────────────────────────────────────────────────────
  W Region
───────────────────────────────────────────────────────────────────────────
  Seed   Team                       R1     R2    S16     E8     FF  Final  Champ
  ────────────────────────────────────────────────────────────────────────
  (1)    Connecticut            100.0%  97.0%  94.0%  86.0% 164.0% 136.0%  48.0%

  (2)    Vanderbilt             100.0%  98.0%  88.0%  59.0%  18.0%   2.0%   0.0%

  (3)    Ohio St                100.0%  98.0%  72.0%  29.0%   8.0%   2.0%   0.0%

  (4)    North Carolina         100.0%  95.0%  52.0%  10.0%  10.0%   0.0%   0.0%

  (5)    Maryland               100.0%  93.0%  48.0%   2.0%   0.0%   0.0%   0.0%

  (6)    Notre Dame             100.0%  86.0%  27.0%   9.0%   0.0%   0.0%   0.0%

  (7)    Illinois               100.0%  74.0%   9.0%   2.0%   0.0%   0.0%   0.0%

  (8)    Iowa St                100.0%  66.0%   3.0%   1.0%   0.0%   0.0%   0.0%
  (9)    Syracuse        

In [43]:
display_championship_odds(sim_w, "Women's Bracket")


  Women's Bracket — Championship Contenders (top 20)
  Seed   Team                   Region       FF  Final  Champ
  ────────────────────────────────────────────────────────────
  (1)    Connecticut            W        164.0% 136.0%  48.0%
  (1)    UCLA                   Z        148.0% 108.0%  23.0%
  (1)    Texas                  Y        172.0%  66.0%  18.0%
  (1)    South Carolina         X        176.0%  58.0%   8.0%
  (2)    LSU                    Z         46.0%  22.0%   2.0%
  (2)    Michigan               Y         10.0%   2.0%   1.0%
  (12)    James Madison          Y          0.0%   0.0%   0.0%
  (16)    SF Austin              Y          0.0%   0.0%   0.0%
  (16)    Missouri St            Y          0.0%   0.0%   0.0%
  (15)    Holy Cross             Y          0.0%   0.0%   0.0%
  (14)    Vermont                Y          0.0%   0.0%   0.0%
  (13)    Miami OH               Y          0.0%   0.0%   0.0%
  (11)    Rhode Island           Y          0.0%   0.0%   0.0%
  (10)  

In [44]:
display_cinderellas(sim_w, "Women's Bracket")


  Women's Bracket — Cinderella Watch (seed >= 10, P(Sweet 16) >= 20%)
  None found at this threshold.
